##Objetivo: responder as cinco perguntas definidas no início do projeto, usando os marts construídos na etapa de modelagem.

In [0]:
%sql
-- perguntas 1 e 2: Qual canal de origem transforma mais leads em vendas? Quanto tempo cada canal leva até fechar?
SELECT * FROM workspace.portfolio_1_olist.mart_canais;

origin,total_leads,negocios_fechados,taxa_conversao_pct,mediana_dias_ate_fechar
nao_identificado,1159,193,16.65,11.0
paid_search,1586,195,12.30,15.0
organic_search,2296,271,11.80,14.5
direct_traffic,499,56,11.22,10.0
referral,284,24,8.45,18.5
social,1350,75,5.56,30.0
display,118,6,5.08,8.5
other_publicities,65,3,4.62,35.0
email,493,15,3.04,21.0
other,150,4,2.67,9.0


In [0]:
%sql
-- pergunta 3: A conversão acontece rápido ou se arrasta? E isso é estável entre os meses?
SELECT * FROM workspace.portfolio_1_olist.mart_coorte_mensal;

mes_contato,total_leads,negocios_fechados,taxa_conversao_pct,mediana_dias_ate_fechar
2017-06-01T00:00:00.000Z,4,0,0.00,null
2017-07-01T00:00:00.000Z,239,2,0.84,398.0
2017-08-01T00:00:00.000Z,386,9,2.33,317.0
2017-09-01T00:00:00.000Z,312,7,2.24,336.0
2017-10-01T00:00:00.000Z,416,14,3.37,246.5
2017-11-01T00:00:00.000Z,445,18,4.04,163.0
2017-12-01T00:00:00.000Z,200,11,5.50,54.0
2018-01-01T00:00:00.000Z,1141,152,13.32,17.5
2018-02-01T00:00:00.000Z,1028,149,14.49,16.0
2018-03-01T00:00:00.000Z,1174,167,14.22,12.5


In [0]:
%sql
-- pergunta 4: O perfil de vendedor que a Olist capta mudou ao longo de 2018?
SELECT
mes_contato,
lead_type,
COUNT(*) AS negocios,
ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY mes_contato), 1) AS pct_no_mes
FROM workspace.portfolio_1_olist.int_funil
WHERE converteu = 1
GROUP BY mes_contato, lead_type
ORDER BY mes_contato, negocios DESC;

mes_contato,lead_type,negocios,pct_no_mes
2017-07-01T00:00:00.000Z,online_small,1,50.0
2017-07-01T00:00:00.000Z,industry,1,50.0
2017-08-01T00:00:00.000Z,online_big,3,33.3
2017-08-01T00:00:00.000Z,online_small,2,22.2
2017-08-01T00:00:00.000Z,online_medium,2,22.2
2017-08-01T00:00:00.000Z,industry,1,11.1
2017-08-01T00:00:00.000Z,offline,1,11.1
2017-09-01T00:00:00.000Z,online_medium,2,28.6
2017-09-01T00:00:00.000Z,offline,1,14.3
2017-09-01T00:00:00.000Z,online_big,1,14.3


In [0]:
%sql
-- pergunta 5: O porte do vendedor se relaciona com ele fabricar ou revender?
SELECT * FROM workspace.portfolio_1_olist.mart_porte_tipo;

lead_type,business_type,negocios,pct_no_porte
industry,manufacturer,69,56.1
industry,reseller,51,41.5
industry,nao_informado,3,2.4
nao_informado,reseller,6,100.0
offline,reseller,81,77.9
offline,manufacturer,21,20.2
offline,nao_informado,2,1.9
online_beginner,reseller,39,68.4
online_beginner,manufacturer,18,31.6
online_big,reseller,103,81.7


## Observações

### Pergunta 1: Qual canal de origem transforma mais leads em vendas?

- Entre os canais identificados, 'paid_search' (12,3%), 'organic_search' (11,8%) e 'direct_traffic' (11,2%) apresentam taxas de conversão superiores à média geral de 10,5%. Em contrapartida, 'social' (5,6%) e 'email' (3,0%) registram os piores desempenhos.

- Destaca-se o caso de 'social', que ocupa a terceira posição em volume de leads (1.350), mas apresenta uma das menores taxas de conversão. Isso indica que maior volume de captação não necessariamente resulta em mais negócios fechados.

- A maior taxa de conversão observada pertence ao grupo 'nao_identificado' (16,7%), formado por leads sem origem registrada. Embora esse grupo não represente um canal acionável e, portanto, não entre nas recomendações de negócio, seu desempenho sugere uma falha de rastreamento que pode estar ocultando fontes relevantes de leads qualificados.

### Pergunta 2: Quanto tempo cada canal leva até fechar?

- A mediana global de fechamento é de 14 dias. Os canais 'paid_search', 'organic_search' e 'direct_traffic' apresentam tempos de fechamento inferiores ou próximos à média, com medianas entre 10 e 15 dias. Em contrapartida, 'social' (30 dias) e 'email' (21 dias) registram os maiores tempos de conversão.

- Esses resultados reforçam o baixo desempenho de 'social' e 'email', que combinam baixas taxas de conversão com ciclos de fechamento mais longos.

- Já os canais 'display', 'other' e 'other_publicities' possuem poucos negócios fechados (entre 3 e 6 casos), tornando suas medianas instáveis e insuficientes para conclusões confiáveis.

### Pergunta 3: A conversão acontece rápido ou se arrasta? E isso é estável entre os meses?

- A conversão pode ser considerada rápida. A mediana do tempo até o fechamento é de 14 dias, e cerca de dois terços dos negócios são concluídos em até 30 dias. Embora existam casos com tempos muito superiores, eles representam uma parcela pequena dos registros.

- Por outro lado, a taxa de conversão não se manteve estável ao longo do período analisado. Em 2017, houve crescimento gradual, saindo de valores próximos de zero até atingir 5,5% no final do ano. Já no primeiro quadrimestre de 2018, a conversão se estabilizou em torno de 13% a 14%.

- O desempenho inferior observado em 2017 dificilmente pode ser atribuído ao tempo de observação, pois essas coortes tiveram mais de um ano para converter até o final da base. Como a maior parte dos negócios fecha em poucas semanas, esse período é mais do que suficiente para capturar praticamente todas as conversões. Assim, as taxas mais baixas de 2017 sugerem uma operação comercial ainda em fase de amadurecimento.Já a leve redução observada na coorte de maio de 2018 deve ser interpretada com cautela. Por ser a última coorte captada, ela dispõe da menor janela de acompanhamento, o que pode ter impedido o registro de parte das conversões mais tardias. Embora a distribuição do tempo até o fechamento sugira que boa parte das conversões já tenha ocorrido, não é possível descartar que a taxa observada esteja subestimada. Portanto, comparações envolvendo essa coorte devem ser feitas com ressalvas.

### Pergunta 4: O perfil de vendedor que a Olist capta mudou ao longo de 2018?

- Não foram observadas mudanças relevantes no perfil dos negócios fechados ao longo de 2018. O tipo de lead mais frequente foi 'online_medium', responsável por cerca de 40% dos negócios, seguido por 'online_big', 'industry' e 'offline'. Essa composição permaneceu relativamente estável entre janeiro e maio.

- A principal oscilação ocorreu em maio, quando a participação de 'online_medium' caiu de aproximadamente 43% para 31%, acompanhada por um aumento moderado dos demais perfis. Ainda assim, a variação não caracteriza uma mudança consistente no perfil dos negócios convertidos.

- Os meses de 2017 apresentam oscilações mais acentuadas, mas o número de negócios fechados nesse período é muito reduzido (entre 2 e 18 por mês), o que limita a confiabilidade dessas diferenças.

- Cabe ressaltar que as informações de perfil estão disponíveis apenas para os 842 negócios fechados. Portanto, esta análise descreve a composição dos leads que converteram, e não do conjunto total de leads captados.

### Pergunta 5: O porte do vendedor se relaciona com ele fabricar ou revender?

- A relação entre porte e tipo de negócio existe, mas o principal fator de diferenciação parece ser o tipo de lead. Em quase todas as categorias, os 'reseller' predominam, representando entre 64% e 82% dos casos.

- A principal exceção é 'industry', no qual os 'manufacturer' são maioria (57,5%). Esse resultado é coerente com a natureza desse segmento, composto por empresas industriais.

- Entre os perfis online, observa-se uma tendência de redução da participação de fabricantes conforme aumenta o porte do lead. Por exemplo, 'online_small' apresenta cerca de 36% de fabricantes, enquanto em 'online_big' essa proporção cai para aproximadamente 18%. No entanto, essa tendência não é uniforme entre todas as categorias e alguns grupos, como 'online_top', possuem poucos registros. Portanto, os resultados sugerem uma associação fraca entre porte e tipo de negócio, insuficiente para uma conclusão mais robusta.